## 1. Setup
Mount Google Drive, clone repository, install all required dependencies.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/AlessandroMarinai/MaskArchitectureAnomaly_CourseProject
%cd /content/MaskArchitectureAnomaly_CourseProject

!pip install -q ood-metrics scikit-learn
!pip install -q "numpy<2"
!pip install -q timm lightning transformers torchmetrics fvcore wandb

!unzip -q /content/drive/MyDrive/MLDL_NewVersionProject/Validation_Dataset.zip -d /content/anomaly_datasets
!ls /content/anomaly_datasets/Validation_Dataset/

Mounted at /content/drive
Cloning into 'MaskArchitectureAnomaly_CourseProject'...
remote: Enumerating objects: 131, done.
remote: Counting objects: 100% (56/56), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 131 (delta 22), reused 19 (delta 19), pack-reused 75 (from 1)
Receiving objects: 100% (131/131), 26.88 MiB | 50.40 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/content/MaskArchitectureAnomaly_CourseProject
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 111.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have nump

## 2. Imports
Core libraries for anomaly segmentation evaluation.
- `ood-metrics`: FPR@95TPR calculation
- `sklearn`: AuPRC (Average Precision)
- `numpy<2`: required for compatibility with torch and ood-metrics

In [ ]:
import os, sys, glob, pickle
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from sklearn.metrics import average_precision_score
from ood_metrics import fpr_at_95_tpr
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, '/content/MaskArchitectureAnomaly_CourseProject/eval')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

input_transform = Compose([
    Resize((512, 1024), Image.BILINEAR),
    ToTensor(),
])
target_transform = Compose([
    Resize((512, 1024), Image.NEAREST),
])
print("✅ Imports done")

Device: cuda
✅ Imports done


## 3. Dataset Paths & Helper Functions
5 anomaly segmentation benchmarks used for evaluation:
- **SMIYC RA-21**: RoadAnomaly21 — animals and objects on road
- **SMIYC RO-21**: RoadObsticle21 — obstacles on road
- **FS L&F**: FishyScapes LostAndFound — small road obstacles
- **FS Static**: FishyScapes Static — pasted anomalous objects
- **Road Anomaly**: General road anomalies

GT processing follows the TA's `evalAnomaly.py` logic exactly.

In [ ]:
BASE = '/content/anomaly_datasets/Validation_Dataset'

DATASETS = {
    'SMIYC_RA21':  f'{BASE}/RoadAnomaly21/images/*.png',
    'SMIYC_RO21':  f'{BASE}/RoadObsticle21/images/*.webp',
    'FS_LnF':      f'{BASE}/FS_LostFound_full/images/*.png',
    'FS_Static':   f'{BASE}/fs_static/images/*.jpg',
    'RoadAnomaly': f'{BASE}/RoadAnomaly/images/*.jpg',
}

def get_gt_path(img_path):
    """Convert image path to GT mask path — from TA's evalAnomaly.py"""
    pathGT = img_path.replace("images", "labels_masks")
    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT and "RoadAnomaly21" not in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    return pathGT

def load_gt(pathGT):
    """Load GT mask — from TA's evalAnomaly.py"""
    mask = Image.open(pathGT)
    mask = target_transform(mask)
    ood_gts = np.array(mask)
    if "RoadAnomaly" in pathGT:
        ood_gts = np.where((ood_gts == 2), 1, ood_gts)
    if "FS_LostFound" in pathGT or "LostAndFound" in pathGT:
        pass  # 0=background, 1=anomaly, 255=ignore
    if "Streethazard" in pathGT:
        ood_gts = np.where((ood_gts == 14), 255, ood_gts)
        ood_gts = np.where((ood_gts < 20), 0, ood_gts)
        ood_gts = np.where((ood_gts == 255), 1, ood_gts)
    return ood_gts

def compute_metrics(anomaly_score_list, ood_gts_list):
    """Compute AuPRC and FPR@95TPR — from TA's evalAnomaly.py"""
    ood_gts = np.array(ood_gts_list)
    anomaly_scores = np.array(anomaly_score_list)
    ood_out = anomaly_scores[ood_gts == 1]
    ind_out = anomaly_scores[ood_gts == 0]
    val_out   = np.concatenate((ind_out, ood_out))
    val_label = np.concatenate((np.zeros(len(ind_out)), np.ones(len(ood_out))))
    auprc = average_precision_score(val_label, val_out) * 100
    fpr95 = fpr_at_95_tpr(val_out, val_label) * 100
    return auprc, fpr95

print("Datasets:")
for name, pattern in DATASETS.items():
    print(f"  {name}: {len(glob.glob(pattern))} images")

Datasets:
  SMIYC_RA21: 10 images
  SMIYC_RO21: 30 images
  FS_LnF: 100 images
  FS_Static: 30 images
  RoadAnomaly: 60 images


In [ ]:
import pickle
import os

LOGITS_DIR = '/content/drive/MyDrive/MLDL_NewVersionProject/saved_logits'

print("Checking saved logit files:")
for f in sorted(os.listdir(LOGITS_DIR)):
    if not f.endswith('.pkl'):
        continue
    path = os.path.join(LOGITS_DIR, f)
    try:
        with open(path, 'rb') as fp:
            data = pickle.load(fp)
        n = len(data['logits'])
        print(f"  ✅ {f}: {n} samples")
    except Exception as e:
        print(f"  ❌ {f}: CORRUPTED — {e} → deleting")
        os.remove(path)

print("\nDone. Re-run Cell 4 to regenerate missing/corrupted files.")

Checking saved logit files:
  ✅ city_FS_LnF.pkl: 99 samples
  ✅ city_FS_Static.pkl: 20 samples
  ✅ city_RoadAnomaly.pkl: 60 samples
  ✅ city_SMIYC_RA21.pkl: 10 samples
  ✅ city_SMIYC_RO21.pkl: 30 samples
  ✅ coco_FS_LnF.pkl: 100 samples
  ✅ coco_FS_Static.pkl: 30 samples
  ✅ coco_RoadAnomaly.pkl: 60 samples
  ✅ coco_SMIYC_RA21.pkl: 10 samples
  ✅ coco_SMIYC_RO21.pkl: 30 samples
  ✅ finetuned_FS_LnF.pkl: 99 samples
  ✅ finetuned_FS_Static.pkl: 20 samples
  ✅ finetuned_RoadAnomaly.pkl: 60 samples
  ✅ finetuned_SMIYC_RA21.pkl: 10 samples
  ✅ finetuned_SMIYC_RO21.pkl: 30 samples

Done. Re-run Cell 4 to regenerate missing/corrupted files.


In [ ]:
# Delete the 0-sample coco files
import os

LOGITS_DIR = '/content/drive/MyDrive/MLDL_NewVersionProject/saved_logits'

files_to_delete = ['coco_FS_LnF.pkl', 'coco_FS_Static.pkl', 'coco_RoadAnomaly.pkl']
for f in files_to_delete:
    path = os.path.join(LOGITS_DIR, f)
    if os.path.exists(path):
        os.remove(path)
        print(f"Deleted: {f}")

print("Done — re-run Cell 4 to regenerate")

Deleted: coco_FS_LnF.pkl
Deleted: coco_FS_Static.pkl
Deleted: coco_RoadAnomaly.pkl
Done — re-run Cell 4 to regenerate


## 4 . Load Models One at a Time + Save Logits to Drive
Load each model, save logits to Drive, delete from memory before loading next.
Logits saved in float16 to minimize RAM. Crash-safe: skips already saved files.

In [ ]:
import wandb, yaml, importlib, warnings, gc, pickle, math, os, sys, glob
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from lightning import seed_everything

seed_everything(0, verbose=False)
os.chdir('/content/MaskArchitectureAnomaly_CourseProject/eomt')
sys.path.insert(0, '/content/MaskArchitectureAnomaly_CourseProject/eomt')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOGITS_DIR = '/content/drive/MyDrive/MLDL_NewVersionProject/saved_logits'
os.makedirs(LOGITS_DIR, exist_ok=True)
print(f"Device: {DEVICE} | Logits dir: {LOGITS_DIR}")

CKPT_CITY     = '/content/drive/MyDrive/MLDL_NewVersionProject/eomt_cityscapes.bin'
CKPT_COCO     = '/content/drive/MyDrive/MLDL_NewVersionProject/eomt_coco.bin'
CKPT_FINETUNE = '/content/drive/MyDrive/MLDL_NewVersionProject/checkpoints_2/finetune_coco_best.bin'

with open("configs/dinov2/cityscapes/semantic/eomt_base_640.yaml") as f:
    config_city = yaml.safe_load(f)
with open("configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml") as f:
    config_coco_cfg = yaml.safe_load(f)

def build_eomt(checkpoint_path, num_classes=19, use_coco_config=False):
    config = config_coco_cfg if use_coco_config else config_city

    state_check = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    if 'model_state_dict' in state_check:
        state_check = state_check['model_state_dict']
    n_tokens   = state_check['network.encoder.backbone.pos_embed'].shape[1]
    patch_size = state_check['network.encoder.backbone.patch_embed.proj.weight'].shape[-1]
    img_size   = int(math.sqrt(n_tokens)) * patch_size
    print(f"  Auto-detected: img_size={img_size}, patch_size={patch_size}")
    del state_check; gc.collect()

    encoder_cfg = config["model"]["init_args"]["network"]["init_args"]["encoder"]
    enc_mod, enc_cls = encoder_cfg["class_path"].rsplit(".", 1)
    encoder_cls = getattr(importlib.import_module(enc_mod), enc_cls)
    enc_init = {k: v for k, v in encoder_cfg.get("init_args", {}).items()}
    enc_init["img_size"]   = (img_size, img_size)
    enc_init["patch_size"] = patch_size
    encoder = encoder_cls(**enc_init)

    net_cfg = config["model"]["init_args"]["network"]
    net_mod, net_cls = net_cfg["class_path"].rsplit(".", 1)
    network_cls = getattr(importlib.import_module(net_mod), net_cls)

    # Prevent config conflicts
    net_kwargs = {k: v for k, v in net_cfg["init_args"].items() if k not in ["encoder", "num_classes"]}
    network = network_cls(
        masked_attn_enabled=False,
        num_classes=num_classes,
        encoder=encoder, **net_kwargs
    )

    warnings.filterwarnings("ignore", message=r".*Attribute 'network'.*")
    lit_mod, lit_cls_name = config["model"]["class_path"].rsplit(".", 1)
    lit_cls = getattr(importlib.import_module(lit_mod), lit_cls_name)

    m_kwargs = {k: v for k, v in config["model"]["init_args"].items() if k not in ["network", "num_classes"]}
    if "stuff_classes" in config.get("data", {}).get("init_args", {}):
        m_kwargs["stuff_classes"] = config["data"]["init_args"]["stuff_classes"]

    model = lit_cls(
        img_size=(img_size, img_size),
        num_classes=num_classes,
        network=network, **m_kwargs
    ).eval().to(DEVICE)
    model.img_size = (img_size, img_size)

    state = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    if 'model_state_dict' in state:
        state = state['model_state_dict']
    missing, unexpected = model.load_state_dict(state, strict=False)
    print(f"  Missing: {len(missing)} | Unexpected: {len(unexpected)}")
    return model

def save_logits_for_model(model, model_name):
    print(f"\n=== {model_name} ===")
    for dataset_name, pattern in DATASETS.items():
        paths = sorted(glob.glob(pattern))
        if not paths:
            print(f"  {dataset_name}: no images found")
            continue

        save_path = f"{LOGITS_DIR}/{model_name}_{dataset_name}.pkl"
        tmp_path  = save_path + '.tmp'
        if os.path.exists(tmp_path) and not os.path.exists(save_path):
            os.rename(tmp_path, save_path)
            print(f"  {dataset_name}: restored from crash ✅")

        if os.path.exists(save_path):
            try:
                with open(save_path, 'rb') as f:
                    data = pickle.load(f)
                if len(data.get('logits', [])) > 0:
                    print(f"  {dataset_name}: already saved ({len(data['logits'])} samples) ✅")
                    continue
                else:
                    print(f"  {dataset_name}: 0 samples — reprocessing")
                    os.remove(save_path)
            except:
                print(f"  {dataset_name}: corrupted — reprocessing")
                os.remove(save_path)

        logits_list, gts_list = [], []
        print(f"  {dataset_name}: {len(paths)} images")

        for idx, path in enumerate(paths):
            image = input_transform(Image.open(path).convert('RGB'))
            image_uint8 = (image * 255).byte()

            with torch.no_grad(), torch.amp.autocast('cuda'):
                imgs      = [image_uint8.to(DEVICE)]
                img_sizes = [image_uint8.shape[-2:]]
                crops, origins = model.window_imgs_semantic(imgs)
                mask_l, class_l = model(crops)
                mask_logits = F.interpolate(mask_l[-1], model.img_size, mode='bilinear')
                per_pixel   = model.to_per_pixel_logits_semantic(mask_logits, class_l[-1])
                logits_full = model.revert_window_logits_semantic(per_pixel, origins, img_sizes)[0]

            logits_r = F.interpolate(
                logits_full.unsqueeze(0), size=(512, 1024), mode='bilinear'
            ).squeeze(0).cpu().half().numpy()

            del logits_full, per_pixel, mask_logits, mask_l, class_l, crops
            torch.cuda.empty_cache()

            pathGT = get_gt_path(path)
            try:
                gt = load_gt(pathGT)
            except:
                del logits_r; continue

            # Skip if no anomaly pixels (label 1) are present in the ground truth
            if 1 not in np.unique(gt):
                del logits_r; continue

            logits_list.append(logits_r)
            gts_list.append(gt)

            # FIX 2: checkpoint every 10 images but DO NOT clear lists
            if (idx + 1) % 10 == 0:
                with open(tmp_path, 'wb') as f:
                    pickle.dump({'logits': logits_list, 'gts': gts_list}, f)
                print(f"    [{idx+1}/{len(paths)}] checkpoint saved")

        # Final save
        with open(save_path, 'wb') as f:
            pickle.dump({'logits': logits_list, 'gts': gts_list}, f)
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        print(f"    ✅ {len(logits_list)} samples saved")
        del logits_list, gts_list; gc.collect()

# Process one model at a time
print("Loading EoMT-Cityscapes...")
model = build_eomt(CKPT_CITY, num_classes=19, use_coco_config=False)
save_logits_for_model(model, 'city')
del model; torch.cuda.empty_cache(); gc.collect()
print("✅ city done\n")

print("Loading EoMT-COCO...")
model = build_eomt(CKPT_COCO, num_classes=133, use_coco_config=True)
save_logits_for_model(model, 'coco')
del model; torch.cuda.empty_cache(); gc.collect()
print("✅ coco done\n")

print("Loading EoMT-Finetuned...")
model = build_eomt(CKPT_FINETUNE, num_classes=19, use_coco_config=True) # Changed num_classes to 19, kept use_coco_config=True
save_logits_for_model(model, 'finetuned')
del model; torch.cuda.empty_cache(); gc.collect()
print("✅ finetuned done\n")

print("✅ All logits saved!")

Device: cuda | Logits dir: /content/drive/MyDrive/MLDL_NewVersionProject/saved_logits
Loading EoMT-Cityscapes...
  Auto-detected: img_size=1024, patch_size=16


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

  Missing: 0 | Unexpected: 0

=== city ===
  SMIYC_RA21: already saved (10 samples) ✅
  SMIYC_RO21: already saved (30 samples) ✅
  FS_LnF: already saved (99 samples) ✅
  FS_Static: already saved (20 samples) ✅
  RoadAnomaly: already saved (60 samples) ✅
✅ city done

Loading EoMT-COCO...
  Auto-detected: img_size=640, patch_size=16
  Missing: 0 | Unexpected: 0

=== coco ===
  SMIYC_RA21: already saved (10 samples) ✅
  SMIYC_RO21: already saved (30 samples) ✅
  FS_LnF: 100 images
    [10/100] checkpoint saved
    [20/100] checkpoint saved
    [30/100] checkpoint saved
    [40/100] checkpoint saved
    [50/100] checkpoint saved
    [60/100] checkpoint saved
    [70/100] checkpoint saved
    [80/100] checkpoint saved
    [90/100] checkpoint saved
    [100/100] checkpoint saved
    ✅ 99 samples saved
  FS_Static: 30 images
    [20/30] checkpoint saved
    [30/30] checkpoint saved
    ✅ 20 samples saved
  RoadAnomaly: 60 images
    [10/60] checkpoint saved
    [20/60] checkpoint saved
    [3

## 5. Post-hoc Methods: MSP, MaxLogit, MaxEntropy, RbA
Apply all 4 methods on saved logits instantly — no model re-run needed.

| Method | Formula | Notes |
|---|---|---|
| MSP | 1 - max(softmax) | Works on any model |
| MaxLogit | -max(logit) | Raw logit score |
| MaxEntropy | -Σ p·log(p) | Uncertainty measure |
| RbA | -max(softmax[:-1]) | Mask architecture specific — excludes void class |

In [ ]:
import pickle

LOGITS_DIR = '/content/drive/MyDrive/MLDL_NewVersionProject/saved_logits'

def apply_methods(model_name, dataset_name, temperature=1.0):
    save_path = f"{LOGITS_DIR}/{model_name}_{dataset_name}.pkl"
    if not os.path.exists(save_path):
        return None

    with open(save_path, 'rb') as f:
        data = pickle.load(f)

    logits_list = data['logits']
    gts_list    = data['gts']
    if not logits_list:
        return None

    msp_s, ml_s, me_s, rba_s = [], [], [], []

    for logits_np in logits_list:
        # Convert float16 back to float32
        logits_np = logits_np.astype(np.float32)
        logits_t  = torch.tensor(logits_np / temperature).float()
        softmax   = F.softmax(logits_t, dim=0).numpy()

        msp_s.append(1.0 - np.max(softmax, axis=0))
        ml_s.append(-np.max(logits_np, axis=0))                       # FIX: removed "/ temperature" — it only rescales every score by 1/T, which can't change AuPRC/FPR95 (ranking metrics). MaxLogit is T-invariant by definition.
        me_s.append(-np.sum(softmax * np.log(softmax + 1e-8), axis=0))
        rba_s.append(-np.sum(np.tanh(logits_np[:-1]), axis=0))        # FIX: real RbA = -Σ_c tanh(ℓ_c) summed over known classes. The old "-max(softmax[:-1])" was just MSP shifted by a constant, so RbA duplicated MSP.

    return {
        'MSP':        compute_metrics(msp_s,  gts_list),
        'MaxLogit':   compute_metrics(ml_s,   gts_list),
        'MaxEntropy': compute_metrics(me_s,   gts_list),
        'RbA':        compute_metrics(rba_s,  gts_list),
    }

DS_LIST     = ['SMIYC_RA21','SMIYC_RO21','FS_LnF','FS_Static','RoadAnomaly']
MODEL_NAMES = ['city','coco','finetuned']

all_results = {}
for model_name in MODEL_NAMES:
    print(f"\n=== {model_name} ===")
    all_results[model_name] = {}
    for ds in DS_LIST:
        r = apply_methods(model_name, ds)
        if r:
            all_results[model_name][ds] = r
            print(f"  {ds}:")
            for m,(a,f) in r.items():
                print(f"    {m:<12}: AuPRC={a:.2f}%  FPR95={f:.2f}%")
        else:
            print(f"  {ds}: no logits found")

print("\n✅ All methods evaluated")


=== city ===
  SMIYC_RA21:
    MSP         : AuPRC=69.61%  FPR95=14.10%
    MaxLogit    : AuPRC=69.47%  FPR95=14.38%
    MaxEntropy  : AuPRC=70.01%  FPR95=14.35%
    RbA         : AuPRC=66.47%  FPR95=95.51%
  SMIYC_RO21:
    MSP         : AuPRC=84.27%  FPR95=4.49%
    MaxLogit    : AuPRC=84.11%  FPR95=4.61%
    MaxEntropy  : AuPRC=84.30%  FPR95=4.62%
    RbA         : AuPRC=81.72%  FPR95=99.85%
  FS_LnF:
    MSP         : AuPRC=16.85%  FPR95=10.18%
    MaxLogit    : AuPRC=16.79%  FPR95=9.94%
    MaxEntropy  : AuPRC=19.59%  FPR95=9.93%
    RbA         : AuPRC=16.06%  FPR95=5.89%
  FS_Static:
    MSP         : AuPRC=62.36%  FPR95=39.22%
    MaxLogit    : AuPRC=62.47%  FPR95=40.20%
    MaxEntropy  : AuPRC=59.84%  FPR95=40.08%
    RbA         : AuPRC=58.33%  FPR95=76.00%
  RoadAnomaly:
    MSP         : AuPRC=67.97%  FPR95=25.53%
    MaxLogit    : AuPRC=67.46%  FPR95=25.44%
    MaxEntropy  : AuPRC=70.03%  FPR95=25.13%
    RbA         : AuPRC=66.66%  FPR95=21.53%

=== coco ===
  SMIYC_RA21

## 6. Temperature Scaling
Temperature T scales logits before softmax: softmax(logits/T).
- **T < 1** → sharper predictions
- **T > 1** → softer predictions, better anomaly detection

Testing T ∈ {0.5, 0.75, 1.1} on EoMT-Cityscapes using saved logits.

In [ ]:
import os, sys, glob, pickle
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from sklearn.metrics import average_precision_score
from ood_metrics import fpr_at_95_tpr
import pandas as pd

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LOGITS_DIR = '/content/drive/MyDrive/MLDL_NewVersionProject/saved_logits'

input_transform = Compose([Resize((512, 1024), Image.BILINEAR), ToTensor()])
target_transform = Compose([Resize((512, 1024), Image.NEAREST)])

DS_LIST  = ['SMIYC_RA21','SMIYC_RO21','FS_LnF','FS_Static','RoadAnomaly']
DS_LABEL = ['SMIYC RA-21','SMIYC RO-21','FS L&F','FS Static','Road Anomaly']
MODEL_NAMES = ['city', 'coco', 'finetuned']
TEMPERATURES = [0.5, 0.75, 1.1]

def compute_metrics(anomaly_score_list, ood_gts_list):
    ood_gts = np.array(ood_gts_list)
    anomaly_scores = np.array(anomaly_score_list)
    ood_out = anomaly_scores[ood_gts == 1]
    ind_out = anomaly_scores[ood_gts == 0]
    val_out   = np.concatenate((ind_out, ood_out))
    val_label = np.concatenate((np.zeros(len(ind_out)), np.ones(len(ood_out))))
    auprc = average_precision_score(val_label, val_out) * 100
    fpr95 = fpr_at_95_tpr(val_out, val_label) * 100
    return auprc, fpr95

def apply_methods(model_name, dataset_name, temperature=1.0):
    save_path = f"{LOGITS_DIR}/{model_name}_{dataset_name}.pkl"
    if not os.path.exists(save_path):
        return None
    with open(save_path, 'rb') as f:
        data = pickle.load(f)
    logits_list = data['logits']
    gts_list    = data['gts']
    if not logits_list:
        return None
    msp_s, ml_s, me_s, rba_s = [], [], [], []
    for logits_np in logits_list:
        logits_np    = logits_np.astype(np.float32)
        scaled       = logits_np / temperature      # apply temperature
        logits_t     = torch.tensor(scaled).float()
        softmax      = F.softmax(logits_t, dim=0).numpy()
        msp_s.append(1.0 - np.max(softmax, axis=0))
        ml_s.append(-np.max(scaled, axis=0))
        me_s.append(-np.sum(softmax * np.log(softmax + 1e-8), axis=0))
        rba_s.append(-np.max(softmax[:-1], axis=0))
    return {
        'MSP':        compute_metrics(msp_s,  gts_list),
        'MaxLogit':   compute_metrics(ml_s,   gts_list),
        'MaxEntropy': compute_metrics(me_s,   gts_list),
        'RbA':        compute_metrics(rba_s,  gts_list),
    }

# Rebuild all_results from saved logits
all_results = {}
for model_name in MODEL_NAMES:
    all_results[model_name] = {}
    for ds in DS_LIST:
        r = apply_methods(model_name, ds, temperature=1.0)
        if r:
            all_results[model_name][ds] = r

print("✅ all_results rebuilt from saved logits")

# Temperature scaling
temp_results = {}
for T in TEMPERATURES:
    print(f"\n=== T={T} ===")
    temp_results[T] = {}
    for ds in DS_LIST:
        r = apply_methods('city', ds, temperature=T)
        if r:
            temp_results[T][ds] = r['MSP']
            a, f = r['MSP']
            print(f"  {ds}: AuPRC={a:.2f}%  FPR95={f:.2f}%")

# Best temperature
best_temp_results = {}
for ds in DS_LIST:
    best_t, best_auprc = None, -1
    for T in TEMPERATURES:
        try:
            auprc = temp_results[T][ds][0]
            if auprc > best_auprc:
                best_auprc, best_t = auprc, T
        except:
            pass
    if best_t:
        r = apply_methods('city', ds, temperature=best_t)
        if r:
            best_temp_results[ds] = (r['MSP'], best_t)

print("\n✅ Temperature scaling done")

✅ all_results rebuilt from saved logits

=== T=0.5 ===
  SMIYC_RA21: AuPRC=69.45%  FPR95=14.10%
  SMIYC_RO21: AuPRC=84.29%  FPR95=4.43%
  FS_LnF: AuPRC=16.84%  FPR95=10.20%
  FS_Static: AuPRC=62.35%  FPR95=39.23%
  RoadAnomaly: AuPRC=67.79%  FPR95=25.56%

=== T=0.75 ===
  SMIYC_RA21: AuPRC=69.55%  FPR95=14.11%
  SMIYC_RO21: AuPRC=84.27%  FPR95=4.47%
  FS_LnF: AuPRC=16.84%  FPR95=10.18%
  FS_Static: AuPRC=62.36%  FPR95=39.23%
  RoadAnomaly: AuPRC=67.90%  FPR95=25.54%

=== T=1.1 ===
  SMIYC_RA21: AuPRC=69.63%  FPR95=14.10%
  SMIYC_RO21: AuPRC=84.27%  FPR95=4.50%
  FS_LnF: AuPRC=16.85%  FPR95=10.18%
  FS_Static: AuPRC=62.36%  FPR95=39.22%
  RoadAnomaly: AuPRC=68.00%  FPR95=25.53%

✅ Temperature scaling done
